# M³ fast: all PANGAEA and iNaturalist datasets, Qwen 3.5 122B

This notebook runs the **two-agent, one-revision-round** M³ path with SURF `Sehyo/Qwen3.5-122B-A10B-NVFP4` on every CSV in the PANGAEA and iNaturalist experiment directories.

PANGAEA uses `pangaea_descriptive_technical_standard` (name, description, keywords, spatial, and temporal). iNaturalist uses `croissant_inaturalist_standard` (spatial and temporal only).

Results are written to `outputs/m3_fast_all_Qwen3.5-122B-A10B-NVFP4`. The Llama, Qwen 3.6, and Devstral output folders are not read or written. Rerunning this notebook skips successes already stored here. Failures stay retryable. Set `M3_MAX_NEW_RUNS=0` to refresh the manifest without new model calls.

In [1]:
import hashlib
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the metadata_agent checkout')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.orchestrator.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS, get_schema_for_standard

MODEL = 'Sehyo/Qwen3.5-122B-A10B-NVFP4'
TOPOLOGY = 'fast'
MAX_NEW_RUNS = int(os.getenv('M3_MAX_NEW_RUNS', '-1'))  # -1 runs all pending files
PANGAEA_STANDARD = 'pangaea_descriptive_technical_standard'
INAT_STANDARD = 'croissant_inaturalist_standard'
PANGAEA_FIELDS = ('name', 'description', 'keywords', 'spatialCoverage',
                  'temporalCoverage', 'spatial', 'temporal')
INAT_FIELDS = ('spatialCoverage', 'temporalCoverage')
DATA_DIRS = {
    'pangaea': ROOT / 'data/experiment/pangaea_datasets',
    'inaturalist': ROOT / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
}
OUT = ROOT / 'outputs/m3_fast_all_Qwen3.5-122B-A10B-NVFP4'
OUT.mkdir(parents=True, exist_ok=True)

for corpus, data_dir in DATA_DIRS.items():
    if not data_dir.is_dir():
        raise FileNotFoundError(data_dir)
if MODEL != 'Sehyo/Qwen3.5-122B-A10B-NVFP4' or TOPOLOGY != 'fast':
    raise ValueError('This notebook is pinned to Sehyo/Qwen3.5-122B-A10B-NVFP4 and the fast topology')
assert len(get_schema_for_standard(PANGAEA_STANDARD).model_fields) == 7
assert len(get_schema_for_standard(INAT_STANDARD).model_fields) == 2
print('Project:', ROOT)
print('Model:', MODEL)
print('Output:', OUT)
print('New M³ runs allowed this session:', 'all pending' if MAX_NEW_RUNS < 0 else MAX_NEW_RUNS)
print('SURF credentials present:', bool(os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')))


Project: /home/com3dian/Github/metadata_agent
Model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Output: /home/com3dian/Github/metadata_agent/outputs/m3_fast_all_Qwen3.5-122B-A10B-NVFP4
New M³ runs allowed this session: all pending
SURF credentials present: True


In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)


def m3_result_path(root: Path, corpus: str, stem: str) -> Path:
    return root / 'm3' / corpus / f'{stem}.json'


def valid_m3_success(path: Path, item: dict, corpus: str) -> bool:
    if not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    return (record.get('status') == 'ok'
            and record.get('model') == MODEL
            and record.get('topology') == TOPOLOGY
            and record.get('standard') == standard
            and record.get('source_sha256') == item['source_sha256']
            and isinstance(record.get('metadata'), dict)
            and set(fields).issubset(record['metadata']))


manifest = {'model': MODEL, 'topology': TOPOLOGY, 'datasets': {}}
for corpus, data_dir in DATA_DIRS.items():
    selected = []
    for source in sorted(data_dir.glob('*.csv')):
        selected.append({'id': source.stem, 'source_sha256': sha256_file(source)})
    if not selected:
        raise FileNotFoundError(f'No CSV files in {data_dir}')
    manifest['datasets'][corpus] = selected
write_json_atomic(OUT / 'manifest.json', manifest)

for corpus, selected in manifest['datasets'].items():
    ready = sum(valid_m3_success(m3_result_path(OUT, corpus, item['id']), item, corpus)
                for item in selected)
    print(corpus, len(selected), 'datasets;', ready, 'successes already saved')
print('Manifest:', OUT / 'manifest.json')


pangaea 75 datasets; 0 successes already saved
inaturalist 100 datasets; 0 successes already saved
Manifest: /home/com3dian/Github/metadata_agent/outputs/m3_fast_all_Qwen3.5-122B-A10B-NVFP4/manifest.json


In [3]:
import logging

log = logging.getLogger()
log.setLevel(logging.INFO)
for handler in log.handlers[:]:
    log.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
log.addHandler(handler)


def run_m3_one(corpus: str, item: dict) -> dict:
    source = DATA_DIRS[corpus] / f"{item['id']}.csv"
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    base = {'dataset_id': item['id'], 'csv': str(source), 'source_sha256': item['source_sha256'],
            'method': 'M3_fast', 'provider': 'surf', 'model': MODEL,
            'topology': TOPOLOGY, 'standard': standard,
            'finished_utc': datetime.now(timezone.utc).isoformat()}
    started = time.monotonic()
    orchestrator = None
    try:
        orchestrator = Orchestrator(topology_name=TOPOLOGY, model_name=MODEL, provider='surf')
        result = orchestrator.run(
            source=str(source), metadata_standard=METADATA_STANDARDS[standard],
            metadata_standard_name=standard, name=item['id'],
        )
        metadata = result.final_metadata if result is not None else None
        if result is None or not result.success:
            raise RuntimeError((result.error if result is not None else 'Planning failed') or 'Run failed')
        if not isinstance(metadata, dict) or not set(fields).issubset(metadata):
            raise ValueError(f'Missing requested metadata fields: {sorted(set(fields)-set(metadata or {}))}')
        metadata = get_schema_for_standard(standard).model_validate(metadata).model_dump(
            mode='json', by_alias=True
        )
        return {**base, 'status': 'ok', 'metadata': metadata,
                'plan_steps_count': result.plan_steps_count,
                'steps_completed': result.steps_completed,
                'trace_path': result.trace_path,
                'runtime_seconds': round(time.monotonic()-started, 2)}
    except Exception as exc:
        return {**base, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
                'trace_path': getattr(orchestrator, 'last_trace_path', None),
                'runtime_seconds': round(time.monotonic()-started, 2)}


if MAX_NEW_RUNS != 0 and not (os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')):
    raise RuntimeError('SURF_API_KEY and SURF_API_BASE are required for new M³ runs')

new_runs = 0
for corpus, selected in manifest['datasets'].items():
    for item in tqdm(selected, desc=f'M³ fast {corpus}'):
        path = m3_result_path(OUT, corpus, item['id'])
        if valid_m3_success(path, item, corpus):
            continue
        if 0 <= MAX_NEW_RUNS <= new_runs:
            continue
        print(f'Starting M³ {corpus} {item["id"]}', flush=True)
        record = run_m3_one(corpus, item)
        write_json_atomic(path, record)
        new_runs += 1
        print(corpus, item['id'], record['status'],
              record.get('error', ''), 'trace:', record.get('trace_path'))
print('New M³ attempts:', new_runs)
for corpus, selected in manifest['datasets'].items():
    print(corpus, 'saved successes:',
          sum(valid_m3_success(m3_result_path(OUT, corpus, item['id']), item, corpus)
              for item in selected), '/', len(selected))


M³ fast pangaea:   0%|          | 0/75 [00:00<?, ?it/s]

Starting M³ pangaea 897882
2026-10-16 21:57:39,789 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-16 21:57:39,790 - INFO - root -   Players per step: 2
2026-10-16 21:57:39,790 - INFO - root -   Debate rounds: 1
2026-10-16 21:57:39,791 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-16 21:57:39,791 - INFO - root - Orchestrator initialized with topology: fast
2026-10-16 21:57:39,792 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/897882_ef4aafde28a84081a54a34028d74e3e8.jsonl
2026-10-16 21:57:39,793 - INFO - root - ============================================================
2026-10-16 21:57:39,793 - INFO - root - STARTING ORCHESTRATION
2026-10-16 21:57:39,794 - INFO - root - Context: 897882
2026-10-16 21:57:39,794 - INFO - root - Type: single_csv
2026-10-16 21:57:39,794 - INFO - root - Resources: ['

M³ fast inaturalist:   0%|          | 0/100 [00:00<?, ?it/s]

Starting M³ inaturalist acianthus_fornicatus
2026-10-17 01:32:37,423 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-17 01:32:37,423 - INFO - root -   Players per step: 2
2026-10-17 01:32:37,424 - INFO - root -   Debate rounds: 1
2026-10-17 01:32:37,424 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-17 01:32:37,424 - INFO - root - Orchestrator initialized with topology: fast
2026-10-17 01:32:37,425 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/acianthus_fornicatus_7844159d38c44565968a978ae8cedf0c.jsonl
2026-10-17 01:32:37,425 - INFO - root - ============================================================
2026-10-17 01:32:37,425 - INFO - root - STARTING ORCHESTRATION
2026-10-17 01:32:37,426 - INFO - root - Context: acianthus_fornicatus
2026-10-17 01:32:37,426 - INFO - root - Type: single_csv
2026-10

/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H2KJ" included an un-recognized timezone "KJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")


...

2026-10-17 02:24:06,349 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:24:06,383 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 4 — model requested 3 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent
2026-10-17 02:24:12,292 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:24:12,294 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop finished after 7 iteration(s), 6 tool invocation(s) — model returned final analysis
2026-10-17 02:24:12,294 - INFO - root - Player 'spatial_temporal_specialist_1': using LLM tool path — 6 tool result(s), 6 trace entry/entries
2026-10-17 02:24:12,295 - INFO - root -   Player 'spatial_temporal_specialist_1' completed execution
2026-10-17 02:24:12,295 - INFO - root -     Output: I now have all the required tool results. Let me summarize the findings:  ## Spa

/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H2KJ" included an un-recognized timezone "KJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H2KJ" included an un-recognized timezone "KJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")


2026-10-17 02:24:50,002 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 1 — model requested 2 tool(s): detect_temporal_columns, detect_spatial_columns
2026-10-17 02:24:54,290 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:24:54,327 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-17 02:24:54,990 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:24:55,029 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-17 02:25:03,724 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-

/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PJ" included an un-recognized timezone "PJ". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PH" included an un-recognized timezone "PH". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
/home/com3dian/Github/metadata_agent/src/tools/context_tools.py:441: FutureWarning: Parsed string "5H9PJ" included an un-recognized timezone "PJ". Dropping unrecognized timezones

2026-10-17 02:28:44,169 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:28:44,215 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-17 02:28:44,247 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:28:44,283 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 4 tool(s): get_temporal_extent, get_temporal_extent, get_temporal_extent, get_spatial_extent
2026-10-17 02:28:53,487 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-17 02:28:53,489 - INFO - root - Player 'spatial_temporal_specialist_2': LLM tool loop finished after 3 iteration(s), 6 tool invocation(s) — model returned final analysis
2026-10-17 02